In [ ]:
!pip install figuard --upgrade -q
print("✓ FiGuard installed")

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient(
    api_key="sb_live_demo",
    base_url="https://sandbox.figuard.io",
)

budget = client.create_budget(
    user_id="agent_001",
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
    authorization_expiry_seconds=300,
    intent_context="travel booking session",
    allocations=[
        {"category": "flight", "limit": 300.00, "enforcement_mode": "CATEGORY_CONSTRAINED", "allowed_categories": ["flight"]},
        {"category": "hotel",  "limit": 200.00, "enforcement_mode": "CATEGORY_CONSTRAINED", "allowed_categories": ["hotel"]},
    ],
)

print(f"✓ Budget created: {budget.id}")
print(f"  Total limit:   ${budget.total_limit}")
print(f"  Allocations:   $300 flights · $200 hotels")

In [ ]:
# Flight — within allocation
auth = client.authorize(
    session_token=budget.session_token,
    agent_id="travel_agent",
    action_type="PURCHASE",
    description="JetBlue SFO→JFK roundtrip",
    requested_quantity=270.00,
    claimed_category="flight",
    idempotency_key="booking-001",
)
print(f"Flight booking:  {auth.decision} — ${auth.approved_quantity}")
client.confirm_event(auth.event_id, confirmed_quantity=267.00)
print("✓ Confirmed.")

# Hotel — within allocation
auth2 = client.authorize(
    session_token=budget.session_token,
    agent_id="travel_agent",
    action_type="PURCHASE",
    description="Marriott 2 nights",
    requested_quantity=180.00,
    claimed_category="hotel",
    idempotency_key="booking-003",
)
print(f"Hotel booking:   {auth2.decision} — ${auth2.approved_quantity}")
client.confirm_event(auth2.event_id, confirmed_quantity=180.00)
print("✓ Confirmed.")

# Flight — exceeds remaining flight allocation ($300 - $267 = $33 left)
auth3 = client.authorize(
    session_token=budget.session_token,
    agent_id="travel_agent",
    action_type="PURCHASE",
    description="Return flight upgrade",
    requested_quantity=150.00,
    claimed_category="flight",
    idempotency_key="booking-004",
)
print(f"Flight upgrade:  {auth3.decision} — {auth3.denial_reason}")
print("Nothing moved. Category limit enforced.")

print(f"\n→ See the spend tree: https://sandbox.figuard.io/ui")